# Ejercicio 5: incorporación de los datos de 2024

Después de ejecutar `scripts/download_data.py --years 2024 2026` los Parquet de 2024 quedaron en `data/raw/<tipo>/2024/` junto a los de 2026. Aquí comprobamos con DuckDB que ahora se pueden consultar los dos años en conjunto y revisamos si las consultas anteriores necesitaban cambios. Las consultas están en `sql/ejercicio_05/` y la discusión en `docs/ejercicio_05.md`.

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 45)

SQL = Path("/workspace/sql")
con = duckdb.connect()  # en memoria, no se guarda ninguna tabla
# Los percentiles exactos sobre 28 millones de filas necesitan memoria, y el
# contenedor comparte unos 5 GB con Metabase, asi que limitamos hilos y memoria.
con.execute("SET threads = 4")
con.execute("SET memory_limit = '2GB'")
# No se define la variable anios: las vistas leen todos los anios que haya en data/raw.
for archivo in ["ejercicio_03/00_vista_viajes.sql", "ejercicio_04/00_vistas_apoyo.sql"]:
    con.execute((SQL / archivo).read_text(encoding="utf-8"))

def correr(carpeta, nombre):
    """Muestra el SQL del archivo y devuelve su resultado."""
    texto = (SQL / carpeta / f"{nombre}.sql").read_text(encoding="utf-8")
    print(texto)
    return con.sql(texto).df()

## 5.5 Archivos y registros por año

Primero confirmamos que los archivos de 2024 están y que su cantidad coincide con lo esperado, 12 por tipo.

In [2]:
correr('ejercicio_05', '5_1_archivos_y_registros_por_anio')

-- 5.5 Verificar que los archivos de 2024 fueron incorporados.
-- Objetivo: contar archivos y registros por tipo y anio sin cargar datos. Los
-- registros salen del pie de pagina de cada Parquet, asi que es inmediato.
-- Fuente: data/raw/*/*/*.parquet (solo la metadata de cada archivo).
SELECT
    regexp_extract(file_name, '(yellow|green)_tripdata', 1) AS tipo,
    CAST(regexp_extract(file_name, 'tripdata_(\d{4})', 1) AS INTEGER) AS anio,
    count(*) AS archivos,
    min(regexp_extract(file_name, '(\d{4}-\d{2})', 1)) AS primer_mes,
    max(regexp_extract(file_name, '(\d{4}-\d{2})', 1)) AS ultimo_mes,
    sum(num_rows) AS viajes
FROM parquet_file_metadata('/workspace/data/raw/*/*/*.parquet')
GROUP BY ALL
ORDER BY tipo, anio;



,tipo,anio,archivos,primer_mes,ultimo_mes,viajes
0,green,2024,12,2024-01,2024-12,660218.0
1,green,2026,8,2026-01,2026-08,337114.0
2,yellow,2024,12,2024-01,2024-12,41169720.0
3,yellow,2026,8,2026-01,2026-08,29703355.0


## 5.6 Viajes por mes, 2024 y 2026 en una sola consulta

Esta es la prueba de que podemos consultar los dos años juntos: una sola consulta sobre todos los archivos, con los meses alineados en columnas.

In [3]:
correr('ejercicio_05', '5_2_registros_por_mes_y_anio')

-- 5.6 Consultar 2024 y 2026 conjuntamente: viajes por mes.
-- Objetivo: ver, en una sola consulta sobre todos los archivos, el volumen de
-- cada mes de cada anio. Los meses se alinean en columnas para poder comparar
-- el mismo mes entre anios. Los meses sin archivo aparecen vacios.
-- Fuente: vista viajes (Parquet de todos los anios en data/raw).
PIVOT (
    SELECT tipo, anio_archivo AS anio, month(pickup) AS mes_del_viaje, count(*) AS viajes
    FROM viajes
    WHERE pickup >= make_date(anio_archivo, 1, 1)
      AND pickup < make_date(anio_archivo + 1, 1, 1)
    GROUP BY ALL
)
ON anio
USING sum(viajes)
ORDER BY tipo, mes_del_viaje;



,tipo,mes_del_viaje,2024,2026
0,green,1,56555.0,40258.0
1,green,2,53578.0,37388.0
2,green,3,57451.0,44203.0
3,green,4,56473.0,44243.0
4,green,5,61007.0,44925.0
5,green,6,54738.0,44156.0
6,green,7,51820.0,41249.0
7,green,8,51806.0,40678.0
8,green,9,54422.0,NaN
9,green,10,56153.0,NaN


## 5.7 Cambios de esquema entre años

Revisamos qué columnas existen en cada año, porque un cambio de columnas o de tipos es lo que más fácil rompe una consulta.

In [4]:
correr('ejercicio_05', '5_3_esquema_por_anio')

-- 5.7 Cambios de esquema entre anios.
-- Objetivo: ver que columnas existen en cada anio y con que tipo fisico de
-- Parquet, para decidir si las consultas necesitan ajustes. Solo se muestran
-- las columnas que no estan en todos los archivos de su tipo, o que cambian de
-- tipo entre anios.
-- Fuente: data/raw/*/*/*.parquet (solo el esquema).
WITH esquema AS (
    SELECT
        regexp_extract(file_name, '(yellow|green)_tripdata', 1) AS tipo,
        CAST(regexp_extract(file_name, 'tripdata_(\d{4})', 1) AS INTEGER) AS anio,
        file_name,
        name AS columna,
        type AS tipo_parquet
    FROM parquet_schema('/workspace/data/raw/*/*/*.parquet')
    WHERE name <> 'schema'
),
archivos AS (
    SELECT tipo, anio, count(DISTINCT file_name) AS total_archivos
    FROM esquema
    GROUP BY ALL
)
SELECT
    e.tipo, e.columna, e.anio, e.tipo_parquet,
    count(DISTINCT e.file_name) AS archivos_con_la_columna,
    a.total_archivos AS archivos_del_anio
FROM esquema e
JOIN archivos a 

,tipo,columna,anio,tipo_parquet,archivos_con_la_columna,archivos_del_anio
0,green,cbd_congestion_fee,2026,DOUBLE,8,8
1,green,request_source,2026,BYTE_ARRAY,3,8
2,yellow,cbd_congestion_fee,2026,DOUBLE,8,8
3,yellow,request_source,2026,BYTE_ARRAY,3,8


## 5.6 Indicadores básicos por año, mismos meses

Una consulta de análisis ejecutada sobre ambos años a la vez. Comparamos enero a agosto de cada año porque 2026 solo llega hasta agosto.

In [5]:
correr('ejercicio_05', '5_4_comparacion_por_anio')

-- 5.6 Consulta conjunta: indicadores basicos por anio, mismos meses.
-- Objetivo: comprobar que las consultas de analisis funcionan sobre varios
-- anios a la vez. Como 2026 solo llega hasta agosto, se comparan enero a agosto
-- de cada anio para que los volumenes sean comparables.
-- Se usan los viajes que pasan las reglas de limpieza de 3_7a.
-- Fuente: vista viajes_limpios (requiere sql/ejercicio_04/00_vistas_apoyo.sql).
SELECT
    tipo,
    anio_archivo AS anio,
    count(*) AS viajes_ene_a_ago,
    round(count(*) * 1.0 / count(DISTINCT CAST(pickup AS DATE))) AS viajes_por_dia,
    round(median(trip_distance), 2) AS mediana_millas,
    round(median(duracion_min), 1) AS mediana_min,
    round(median(total_amount), 2) AS mediana_total,
    round(avg(total_amount), 2) AS total_promedio
FROM viajes_limpios
WHERE month(pickup) <= 8
GROUP BY ALL
ORDER BY tipo, anio;



,tipo,anio,viajes_ene_a_ago,viajes_por_dia,mediana_millas,mediana_min,mediana_total,total_promedio
0,green,2024,412714,1691.0,1.98,12.0,19.15,23.78
1,green,2026,319257,1314.0,2.16,13.4,20.52,25.42
2,yellow,2024,25492918,104479.0,1.80,12.7,21.00,28.22
3,yellow,2026,28146188,115828.0,1.94,14.2,23.58,30.18


## 5.7 Los problemas de calidad de 2026 en 2024

Repetimos las mediciones de calidad del ejercicio 3 por año para ver si los nulos y los montos negativos son propios de 2026.

In [6]:
correr('ejercicio_05', '5_5_calidad_por_anio')

-- 5.7 Los problemas de calidad de 2026, aparecen tambien en 2024?
-- Objetivo: repetir las mediciones principales de 3.6 por anio, para ver si
-- los nulos, las distancias en cero y los montos negativos son estables o son
-- propios de 2026.
-- Fuente: vista viajes (Parquet de todos los anios en data/raw).
SELECT
    tipo,
    anio_archivo AS anio,
    count(*) AS viajes,
    round(100.0 * count(*) FILTER (WHERE passenger_count IS NULL) / count(*), 1) AS pct_pasajeros_nulo,
    round(100.0 * count(*) FILTER (WHERE coalesce(payment_type, 0) = 0) / count(*), 1) AS pct_pago_nulo_o_0,
    round(100.0 * count(*) FILTER (WHERE trip_distance = 0) / count(*), 2) AS pct_distancia_cero,
    round(100.0 * count(*) FILTER (WHERE total_amount < 0) / count(*), 2) AS pct_total_negativo,
    round(100.0 * count(*) FILTER (WHERE epoch(dropoff - pickup) > 6 * 3600) / count(*), 3) AS pct_mas_de_6_horas,
    count(*) FILTER (WHERE pickup < make_date(anio_archivo, 1, 1)
                        OR pickup >

,tipo,anio,viajes,pct_pasajeros_nulo,pct_pago_nulo_o_0,pct_distancia_cero,pct_total_negativo,pct_mas_de_6_horas,pickup_fuera_del_anio,con_cbd_congestion_fee
0,green,2024,660218,3.7,3.7,5.24,0.33,0.398,20,0
1,green,2026,337114,14.5,14.5,3.62,0.30,0.327,14,337114
2,yellow,2024,41169720,9.9,9.9,1.89,1.48,0.053,56,0
3,yellow,2026,29703355,26.0,26.0,3.21,0.54,0.025,17,29703355


## Efecto de la variable `anios`

Las vistas leen todos los años que haya en `data/raw`, y la variable de sesión `anios` permite acotarlos sin tocar ninguna consulta. Lo comprobamos con la misma consulta bajo cuatro configuraciones.

In [7]:
consulta = "SELECT anio_archivo AS anio, count(*) AS viajes FROM viajes GROUP BY 1 ORDER BY 1"
for config in ["RESET VARIABLE anios", "SET VARIABLE anios = [2026]", "SET VARIABLE anios = [2024]", "SET VARIABLE anios = [2024, 2026]"]:
    con.execute(config)
    print(config)
    print(con.sql(consulta).df().to_string(index=False))
    print()
con.execute("RESET VARIABLE anios")

RESET VARIABLE anios


 anio   viajes
 2024 41829938
 2026 30040469

SET VARIABLE anios = [2026]


 anio   viajes
 2026 30040469

SET VARIABLE anios = [2024]


 anio   viajes
 2024 41829938

SET VARIABLE anios = [2024, 2026]


 anio   viajes
 2024 41829938
 2026 30040469

